<a href="https://colab.research.google.com/github/nhongtat-a11y/codefinity-deep-learning-with-tensorflow/blob/master/Assignment_3_1_Single_Layer_Perceptron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3.1-Single Layer Perceptron


## Activity Overview

A neural network (NN) is based on a collection of connected units or nodes called artificial neurons, which loosely model the neurons in a biological brain. The basic unit of artificial neural network is the perceptron. Each perceptron has connections, like the synapses in a biological brain, that can compute and transmit a signal to other units (neurons).

In this notebook, a very simple version of a single perceptron is first built by hand, and used to make a prediction on the Titanic dataset. Then, the implementation of a perceptron using `sklearn` is handled.

This activity is designed to help you apply the machine learning algorithms you have learned using the packages in `Python`. `Python` concepts, instructions, and starter code are embedded within this Jupyter Notebook to help guide you as you progress through the activity. Remember to run the code of each code cell prior to submitting the assignment. Upon completing the activity, we encourage you to compare your work against the solution file to perform a self-assessment.

## Index:

#### Week 3:  Single-Layer Perceptron



- [Part 1](#part1) -  Problem Setup
- [Part 2](#part2) -  Perceptron
- [Part 3](#part3) - Implementing the Perceptron
- [Part 4](#part4) - Single Layer Perceptron in `sklearn`


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Introduction of Dataset
## Titanic Dataset
Dataset provides detailed information about the passengers aboard the ill-fated RMS Titanic. This dataset is commonly used for machine learning and data analysis tasks due to its rich set of features and historical significance.

Dataset Overview
Total Entries: 891
Total Columns: 12

Column Descriptions
PassengerId: Unique identifier for each passenger.

Survived: Survival status (0 = No, 1 = Yes).

Pclass: Passenger class (1 = 1st class, 2 = 2nd class, 3 = 3rd class).

Name: Full name of the passenger.

Sex: Gender of the passenger.

Age: Age of the passenger (some entries are missing).

SibSp: Number of siblings/spouses aboard the Titanic.

Parch: Number of parents/children aboard the Titanic.

Ticket: Ticket number.

Fare: Ticket fare.

Cabin: Cabin number (many entries are missing).

Embarked: Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton).

In [17]:
import numpy as np # linear algebra
np.random.seed(10)
import pandas as pd # data processing, CSV file I/O (e.g., pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

# Required magic to display matplotlib plots in notebooks
%matplotlib inline

#reading the files
data_train = pd.read_csv('sample_data/train.csv')
data_test = pd.read_csv('sample_data/test.csv')

In the code cell below, the function `head()` with argument 4 is used to display the first four rows of the dataset. You should adjust the code below to display eight rows in the dataset.

In [18]:
#data_train.head(4)

data_train.head(8)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S


Some columns are not deemed fit for Machine Learning model. These columns are removed. The selection of columns
plays a very crucial role in the performance of the model. This step is the Data preprocessing step. Without going
in depth about that step, only the colums which are useful in model creation are retained. The columns "PassengerId", "Name", "Ticket", "Cabin", "Embarded", "Age" are removed.

In [19]:
#remove unnecessary columns as they are not helpful in finding if a set of passengers has survived or not
data_train.drop(["PassengerId","Name","Ticket","Cabin","Embarked","Age"],axis=1,inplace=True)
data_test.drop(["PassengerId","Name","Ticket","Cabin","Embarked","Age"],axis=1,inplace=True)

In [20]:
data_test.isnull().sum()

,0
Pclass,0
Sex,0
SibSp,0
Parch,0
Fare,0


In [21]:
data_train.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
SibSp,0
Parch,0
Fare,0


[Back to top](#Index:)

<a id='part2'></a>

## Problem Setup

In this short activity, the basic multi-layer perceptron is manually build to predict if a set of passengers have survived or not.

In the code below, a dictonary `dict_live` is defined, which maps the entries in the column `Survived` from 0 and 1 to `Perished` and `Survived`, respectively.

In [22]:
# A dictionary to transform the 0,1 values in the
# labels to a String that defines the fate of the passenger
dict_live = {
    0 : 'Perished',
    1 : 'Survived'
}

In the code below, fill the ellipsis in the definition of the dictonary `dict_sex` to map the entries in the column `Sex` from `male` and `female` to `0` and `1`, respectively.

In [23]:
# We define a dictionary to assign labels 0 or 1
# corresponding to the sex
dict_sex = {
    'male' : 0,
    'female' : 1
}


# We define a dictionary to assign labels 0 or 1
# corresponding to the sex
dict_sex = {
    'male' : 0,
    'female' : 1
}

Run the code cell below, to create the column `Bsex` with the values in `dict_sex` and append it to the dataframe. After running, you should see this new data as the right-most column in the dataframe.

In [24]:
#We apply the dictionary using a lambda function and the pandas .apply() module
data_train['Bsex'] = data_train['Sex'].apply(lambda x: dict_sex[x])
data_test['Bsex'] = data_test['Sex'].apply(lambda x: dict_sex[x])

#delete old column "Sex"
data_train.drop(["Sex"],axis=1, inplace=True)
data_test.drop(["Sex"],axis=1, inplace=True)

#This line displasy the dataframe with the new column
data_train.head()

,Survived,Pclass,SibSp,Parch,Fare,Bsex
0,0,3,1,0,7.2500,0
1,1,1,1,0,71.2833,1
2,1,3,0,0,7.9250,1
3,1,1,1,0,53.1000,1
4,0,3,0,0,8.0500,0


Before implementing the perpeptron in `Python`, the features used in prediction are initially defined.

In the code cell below, fill the ellipsis with `Pclass` and `BSex` to indicate a list with these two values as our two-dimensional feature vector. Note that we used the function `to_numpy()` to convert our features to `NumPy` arrays.

In [ ]:
#features = data_train[['...', '...']].to_numpy()
#print(features)

# Now the features are a 2 column matrix whose entries are
# the Class (1,2,3) and the Sex (0,1) of the passengers
features = data_train[['Pclass', 'Bsex']].to_numpy()
print(features)

[Back to top](#Index:)

<a id='part3'></a>

# Perceptron

The perceptron is a basic unit or function that mimics the human neuron. It receives a vector (i.e., array) $x_i$ of signals, where $i$ stands for the $i$-th input; then weights each of them by a vector of weights $w_i$. It also adds a *bias*, $w_0$, to shift the decision boundary away from the origin as needed.



Thus, the intermediate value $z$ is given by $z = w_0 + \sum_{i=1}^m w_i x_i = w_0 + \mathbf{w}^T \cdot \mathbf{x}.$

### Activation Functions

The perceptron next ignites an output through an activation function acting on the intermediate value $z$ (note: $z$ is sometimes called the *pre-activation* value, since it is fed to the activation function). Activation functions can vary, but the ones that we will consider here are:


#### The *sigmoid* Function
$$\phi(z) = \frac{1}{1+e^{-z}} \,,$$



#### The Rectified Linear Unit (ReLU)

$$\phi(z) = \mathrm{max}(0, z) \,,$$


### Output
Combining the weighted linear combination (that calculates $z$) with the activation function $\phi$, we see that the output $a$ of the perceptron is given by

$$a = \phi(z) = \phi \left( \sum_{i=1}^{n} w_i \, x_i + w_0   \right) ,$$

or, in vector representation:

$$a = \phi(z) = \phi \left(\mathbf{w}^T \cdot \mathbf{x} + w_0  \right) \,.$$

[Back to top](#Index:)

<a id='part4'></a>

## Implementing the Perceptron

The simple implementation of perceptron is presented below.

First, we need to define two functions in `Python`, `sigmoid_act` and `ReLu_act`, for the activations.

#### `sigmoid_act`

The function `sigmoid_act` takes as argument a vector `z`and returns the output `a`.

In [ ]:
# Define the sigmoid activator
def sigmoid_act(z):
    # sigmoid
    a = 1/(1+ np.exp(-z))
    return a

#### `ReLU_act`

The function `ReLU_act` takes as argument a vector `z`and returns the output using the `max(0,z)`.

In [ ]:
# We may employ the Rectifier Linear Unit (ReLU)
def ReLU_act(z):
    return max(0, z)


In the code cell below, a function, `perceptron`,is defined,  that takes as input an array `X` representing the features in the dataframe and the argument `act` for selecting the activation function.

Run the code cell below.

In [ ]:
def perceptron(X, act):
    np.random.seed(1) #seed for reproducibilty
    shapes = X.shape #get the number of (rows, columns)
    n = shapes[0] + shapes[1] #adding the number of rows and colums
    # Generating random weights and bias
    w = 2*np.random.random(shapes) - 0.5 #we want initial w between -1 and 1
    w_0 = np.random.random(1)

    # Initialize the function
    f = w_0[0]
    for i in range(0, X.shape[0]-1): #run over column elements
        for j in range(0, X.shape[1]-1): #run over rows elements
            f += w[i, j]*X[i,j]/n #adding each component of input multiplied by corresponding weight
    # Pass it to the activation function and return it as an output
    if act == 'Sigmoid':
        output = sigmoid_act(f)
    elif act == "ReLU":
        output = ReLU_act(f)
    return output

An example of an output of the perceptron with the sigmoid activation is given below:

In [ ]:
print('Output with sigmoid activator: ', perceptron(features, act = 'Sigmoid'))

In the code cell below, try to generate the output by using the ReLU function. Simply observe the code above and fill - in the ellippsis with the name of the function that defined the desired activation.

In [ ]:
#print('Output with ReLU activator: ', perceptron(features, act = "..."))

print('Output with ReLU activator: ', perceptron(features, act = "ReLU"))

In [ ]:
##Question:

##The outputs we have obtained are different for the two activation functions. Can you explain why?


**The outputs differ when using these two activation function. This is because in the sigmoidal function, the input values are squashed into a small range, making it useful for binary classification problems. However, it can suffer from the vanishing gradient problem, where gradients become very small for large positive or negative inputs, slowing down the learning process.
In the case of "Relu" activation function, it allows for faster and more efficient training by not saturating for positive values, thus avoiding the vanishing gradient problem. However, it can suffer from the “dying ReLU” problem, where neurons can get stuck during training if they output zero for all inputs.**




[Back to top](#Index:)

<a id='part5'></a>

## Single-Layer Perceptron in `sklearn`

The library `sklearn` offers a built-in implementation of a single-layer perceptron. Even better, the implementation also knows how to calculate gradients of all parts of the perceptron, and use that information to implement back-propagation.

 In this next example, we will same dataset

In [ ]:
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
## Question

## How many features (independent variables) does `X` have?



**There are 5 independent variables. They are 'Pclass', 'SibSp', 'Parch', 'Fare', 'Bsex'. The X_train.columns is used to see the names of columns.**



Run the code cell below to visualize the values of `y` for all of our data.

In [ ]:
# Split the data into 75% training data and 25% test data
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=...)
X_train = data_train.drop(columns=['Survived'])
y_train = data_train['Survived']



## Preprocess the `X` Data by Scaling

It is often standard practice to standardize all the features to have mean equal to zero and unit variance. Standardization of a dataset is a common requirement for many machine learning estimators: they might behave badly if the individual features do not more or less look like standard normally distributed data (e.g., Gaussian with 0 mean and unit variance).

We can accomplish this by calling the function `StandardScaler` and by fitting it over the `X_train` set.

Run the code cell below:

In [ ]:
sc = StandardScaler()
sc.fit(X_train)

The scaler function is applied to both the training and testing datasets.

Run the code cell by filling the ellipsis with the `X_test` set.

In [ ]:
# Apply the scaler to the X training data
X_train_std = sc.transform(X_train)

# Apply the SAME scaler to the X test data
#X_test_std = sc.transform(...)

X_test_std = sc.transform(data_test)

# Train a Single-Layer Perceptron Model

Next, the `Perceptron` classifier is trained on the scaled `X_train` data with the corresponding labels in `Y_train`.

Run the code cell below. Inside the classifier `Perceptron`, set the learning rate `eta0` equal to 0.1 and the `random_state` equal to 1. Note that by default, the sklearn `Perceptron` uses a ReLU activation function.

In [ ]:
# Create a perceptron object with the parameters:max_ieter (epochs) equal to 40 learning rate of 0.1
#ppn = Perceptron(max_iter = 40, eta0 = ..., random_state = ...)

ppn = Perceptron(max_iter = 1000, eta0 = 0.1, random_state = 1)

# Train the perceptron
ppn.fit(X_train_std, y_train)

Now we can use this trained perceptron, `ppn`, to predict the results for the **scaled** test data.

In the code cell below, fill in the ellipsis with the name of the variable corresponding to the set above.

In [ ]:
# Apply the trained perceptron on the X data to make predicts for the y test data
#y_pred = ppn.predict(...)

y_pred = ppn.predict(X_train_std)
y_pred

We can compare the predicted `y` with the true `y`. Run the code cells below to examine these.

Finally, we can evaluate the accuracy of our model by using the function `accuracy_score` from `sklearn`, which uses the formula:

$$\text{accuracy} = 1 - \frac{\text{observations predicted wrong}}{\text{total observations}}$$

Run the code below to evaluate the accuracy of our model.

In [ ]:
# View the accuracy of the model, which is: 1 - (observations predicted wrong / total observations)
print('Accuracy: %.2f' % accuracy_score(y_train, y_pred))

# Train a Multilayer Perceptron (MLP) Model

Finally, a multilayer perceptron (MLP) is trained using the same training and test data. Run the cell below to train our `mlp` model, with three hidden layers each having 10 hidden nodes.

In [ ]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(10, 10, 10), max_iter=1000)
mlp.fit(X_train_std, y_train)

In [ ]:
mlp_y_pred = mlp.predict(X_train_std)
mlp_y_pred

In [ ]:
# View the accuracy of the model, which is: 1 - (observations predicted wrong / total observations)
print('Accuracy: %.2f' % accuracy_score(y_train, mlp_y_pred))

In [ ]:
## Question: Which model, the single-layer perceptron or the multi-layer perceptron, performed better on test data? Why?


**The multi-layer perceptron model is better than single-layer perceptron on the test data because the accuracy of multilayer perceptron is higher than that of single layer perceptron.**